In [0]:
# ──────────────────────────────────────────────────────────────
# CELL 1 │ Configuration
# ──────────────────────────────────────────────────────────────
BRONZE_PATH     = "/Volumes/workspace/fraud_platform/data/bronze/transactions"
SILVER_PATH     = "/Volumes/workspace/fraud_platform/data/silver/transactions"
CHECKPOINT_PATH = "/Volumes/workspace/fraud_platform/data/checkpoints/silver"

# MCC codes that are inherently suspicious regardless of amount
SUSPICIOUS_MCC = ["7995", "5933", "6051"]
# 7995 = gambling, 5933 = pawn shop, 6051 = crypto/money transfer

# Cards are Irish — any transaction outside these is "foreign"
HOME_COUNTRIES = ["IE"]

print("✅ Silver config loaded")
print(f"   Source : {BRONZE_PATH}")
print(f"   Sink   : {SILVER_PATH}")

✅ Silver config loaded
   Source : /Volumes/workspace/fraud_platform/data/bronze/transactions
   Sink   : /Volumes/workspace/fraud_platform/data/silver/transactions


In [0]:
# ──────────────────────────────────────────────────────────────
# CELL 2 │ Read Bronze + compute fraud signals
# ──────────────────────────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_df = spark.read.format("delta").load(BRONZE_PATH)

# ── Window: same card, ordered by time, look back 5 minutes ──
# rangeBetween uses seconds when column is timestamp cast to long
card_5min_window = (
    Window
    .partitionBy("card_id")
    .orderBy(F.col("event_ts").cast("long"))
    .rangeBetween(-300, 0)   # 300 seconds = 5 minutes
)

silver_df = (
    bronze_df

    # ── Signal 1: High amount ──────────────────────────────────
    .withColumn("is_high_amount",
        F.col("amount_local") > 500
    )

    # ── Signal 2: Foreign country ─────────────────────────────
    .withColumn("is_foreign_country",
        ~F.col("country_code").isin(HOME_COUNTRIES)
    )

    # ── Signal 3: Suspicious MCC ──────────────────────────────
    .withColumn("is_suspicious_mcc",
        F.col("merchant_category_code").isin(SUSPICIOUS_MCC)
    )

    # ── Signal 4: Rapid succession (rolling 5-min count) ──────
    .withColumn("txn_count_5min",
        F.count("transaction_id").over(card_5min_window)
    )
    .withColumn("is_rapid_succession",
        F.col("txn_count_5min") > 2
    )

    # ── Risk score: weighted sum of signals (0–100) ────────────
    .withColumn("risk_score",
        (F.col("is_high_amount").cast("int")      * 40) +
        (F.col("is_foreign_country").cast("int")  * 30) +
        (F.col("is_suspicious_mcc").cast("int")   * 20) +
        (F.col("is_rapid_succession").cast("int") * 10)
    )

    # ── Risk label ─────────────────────────────────────────────
    .withColumn("risk_label",
        F.when(F.col("risk_score") >= 60, "HIGH")
         .when(F.col("risk_score") >= 30, "MEDIUM")
         .otherwise("LOW")
    )

    .withColumn("processed_at", F.current_timestamp())
)

print(f"✅ Signals computed on {silver_df.count()} rows")
silver_df.printSchema()

✅ Signals computed on 1786 rows
root
 |-- kafka_partition: integer (nullable = true)
 |-- kafka_offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- transaction_id: string (nullable = true)
 |-- card_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- merchant_id: string (nullable = true)
 |-- merchant_name: string (nullable = true)
 |-- merchant_category_code: string (nullable = true)
 |-- amount_local: double (nullable = true)
 |-- currency_code: string (nullable = true)
 |-- amount_eur: double (nullable = true)
 |-- country_code: string (nullable = true)
 |-- terminal_type: string (nullable = true)
 |-- event_timestamp: long (nullable = true)
 |-- ip_address: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- _fraud_simulation_type: string (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_topic: str

In [0]:
# ──────────────────────────────────────────────────────────────
# CELL 3 │ Write to Silver Delta table
# ──────────────────────────────────────────────────────────────
(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")       # Silver is always rebuilt from Bronze
    .option("overwriteSchema", "true")
    .save(SILVER_PATH)
)

print("🥈 Silver written successfully")
print(f"   Path: {SILVER_PATH}")

🥈 Silver written successfully
   Path: /Volumes/workspace/fraud_platform/data/silver/transactions


In [0]:
# ──────────────────────────────────────────────────────────────
# CELL 4 │ Verify Silver — risk score distribution
# ──────────────────────────────────────────────────────────────
silver = spark.read.format("delta").load(SILVER_PATH)

print(f"Total rows: {silver.count()}\n")

print("── Risk label distribution ──")
silver.groupBy("risk_label").count().orderBy("count", ascending=False).show()

print("── High risk transactions (score ≥ 60) ──")
silver.filter(F.col("risk_score") >= 60) \
    .select("event_ts", "card_id", "merchant_name", "amount_local",
            "country_code", "risk_score", "risk_label",
            "is_high_amount", "is_foreign_country",
            "is_suspicious_mcc", "is_rapid_succession") \
    .orderBy(F.col("risk_score").desc()) \
    .show(20, truncate=False)

Total rows: 1786

── Risk label distribution ──
+----------+-----+
|risk_label|count|
+----------+-----+
|       LOW| 1383|
|    MEDIUM|  396|
|      HIGH|    7|
+----------+-----+

── High risk transactions (score ≥ 60) ──
+-----------------------+---------+----------------+------------+------------+----------+----------+--------------+------------------+-----------------+-------------------+
|event_ts               |card_id  |merchant_name   |amount_local|country_code|risk_score|risk_label|is_high_amount|is_foreign_country|is_suspicious_mcc|is_rapid_succession|
+-----------------------+---------+----------------+------------+------------+----------+----------+--------------+------------------+-----------------+-------------------+
|2026-08-20 19:39:55.848|CARD_0030|Paris Metro Shop|2513.34     |FR          |80        |HIGH      |true          |true              |false            |true               |
|2026-08-20 19:35:48.509|CARD_0004|Marks & Spencer |8391.63     |GB          |80    